<a href="https://colab.research.google.com/github/romero-sebastian/econ3916-statsml/blob/main/Lab17/Lab_17_Predicting_Regressions_with_Log_Regs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression, LogisticRegression
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

import fredapi
FRED_API_KEY = '378ca90c1377309f9817f5701655101a'
fred = fredapi.Fred(api_key=FRED_API_KEY)

spread_raw = fred.get_series('T10Y3M', observation_start='1970-01-01')
recession_raw = fred.get_series('USREC', observation_start='1970-01-01')
USE_FRED = True
print("FRED data loaded ✓")

spread_monthly = spread_raw.resample('ME').last()
recession_monthly = recession_raw.resample('ME').max()

df = pd.DataFrame({
    'yield_spread': spread_monthly,
    'recession': recession_monthly
}).dropna()

df['yield_spread_lag12'] = df['yield_spread'].shift(12)
df = df.dropna()

print(f"Dataset: {df.index[0].strftime('%Y-%m')} to {df.index[-1].strftime('%Y-%m')}")
print(f"Observations: {len(df)}")
print(f"Recession months: {df['recession'].sum()} ({df['recession'].mean():.1%} of sample)")
df.tail()

X = df[['yield_spread_lag12']].values
y = df['recession'].values

lpm_model = LinearRegression()
lpm_model.fit(X, y)

spread_grid = np.linspace(df['yield_spread_lag12'].min() - 0.2,
                           df['yield_spread_lag12'].max() + 0.2, 500).reshape(-1, 1)
lpm_preds = lpm_model.predict(spread_grid)
lpm_fitted = lpm_model.predict(X)

n_below_zero = (lpm_fitted < 0).sum()
n_above_one  = (lpm_fitted > 1).sum()

print(f"LPM fitted — Intercept: {lpm_model.intercept_:.4f}, Slope: {lpm_model.coef_[0]:.4f}")
print(f"Predicted probability < 0: {n_below_zero} ({n_below_zero/len(df):.1%})")
print(f"Predicted probability > 1: {n_above_one} ({n_above_one/len(df):.1%})")

logit_model = LogisticRegression(random_state=42)
logit_model.fit(X, y)

logit_preds = logit_model.predict_proba(spread_grid)[:, 1]
logit_fitted = logit_model.predict_proba(X)[:, 1]

print(f"Intercept (β₀): {logit_model.intercept_[0]:.4f}")
print(f"Slope (β₁):     {logit_model.coef_[0][0]:.4f}")
print(f"Odds ratio (exp(β₁)): {np.exp(logit_model.coef_[0][0]):.4f}")
print(f"Min predicted probability: {logit_fitted.min():.4f}")
print(f"Max predicted probability: {logit_fitted.max():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=False)
COLOR_RECESSION = '#d62728'
COLOR_EXPANSION = '#1f77b4'
COLOR_LPM = '#ff7f0e'
COLOR_LOGIT = '#2ca02c'
recession_mask = y == 1
x_vals = df['yield_spread_lag12'].values

for ax, model_preds, model_name, color in [
    (axes[0], lpm_preds,   'LPM (OLS)',           COLOR_LPM),
    (axes[1], logit_preds, 'Logistic Regression', COLOR_LOGIT)
]:
    ax.scatter(x_vals[~recession_mask], y[~recession_mask], color=COLOR_EXPANSION, alpha=0.3, s=15)
    ax.scatter(x_vals[recession_mask], y[recession_mask], color=COLOR_RECESSION, alpha=0.5, s=15)
    ax.plot(spread_grid.ravel(), model_preds, color=color, lw=2.5)
    ax.axhline(0, color='black', lw=0.8, linestyle='--', alpha=0.5)
    ax.axhline(1, color='black', lw=0.8, linestyle='--', alpha=0.5)
    ax.set_xlabel('Yield Spread, 12-month lag (pp)')
    ax.set_ylabel('P(Recession within 12 months)')
    ax.set_title(model_name, fontweight='bold')
    ax.set_ylim(-0.3, 1.3)

axes[0].fill_between(spread_grid.ravel(), -0.3, 0, color=COLOR_LPM, alpha=0.08)
axes[0].fill_between(spread_grid.ravel(), 1, 1.3, color=COLOR_LPM, alpha=0.08)
plt.tight_layout()
plt.savefig('lpm_vs_logistic.png', dpi=150, bbox_inches='tight')
plt.show()

beta_1 = logit_model.coef_[0][0]

# TODO filled in:
odds_ratio = np.exp(beta_1)

X_sm = sm.add_constant(df[['yield_spread_lag12']])
logit_sm = sm.Logit(df['recession'], X_sm).fit(disp=False)
coef_table = logit_sm.summary2().tables[1]
print("Statsmodels Logit summary:")
print(coef_table)

row = coef_table.loc['yield_spread_lag12']
# TODO filled in:
or_point = np.exp(row['Coef.'])
or_lower = np.exp(row['[0.025'])
or_upper = np.exp(row['0.975]'])

print(f"Odds Ratio: {or_point:.4f}")
print(f"95% CI:     [{or_lower:.4f}, {or_upper:.4f}]")
print(f"A 1pp increase in the yield spread multiplies recession odds by {or_point:.3f}.")

df['recession_prob'] = logit_model.predict_proba(X)[:, 1]

print(f"Min probability: {df['recession_prob'].min():.4f}")
print(f"Max probability: {df['recession_prob'].max():.4f}")
print(f"Mean probability: {df['recession_prob'].mean():.4f}")
print(f"Base rate: {df['recession'].mean():.4f}")
print(df[['yield_spread', 'yield_spread_lag12', 'recession', 'recession_prob']].tail(24).round(4))

df_plot = df['2000':].copy()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True,
                                 gridspec_kw={'height_ratios': [2, 1]})

ax1.plot(df_plot.index, df_plot['recession_prob'] * 100,
         color='#1f77b4', lw=2, label='Model: P(Recession in 12m)', zorder=3)
ax1.axhline(50, color='black', lw=0.8, linestyle='--', alpha=0.5, label='50% threshold')

def get_recession_bands(df_in):
    in_recession = False
    bands = []
    start = None
    for date, val in df_in['recession'].items():
        if val == 1 and not in_recession:
            start = date
            in_recession = True
        elif val == 0 and in_recession:
            bands.append((start, date))
            in_recession = False
    if in_recession:
        bands.append((start, df_in.index[-1]))
    return bands

recession_bands = get_recession_bands(df_plot)

for i, (start, end) in enumerate(recession_bands):
    ax1.axvspan(start, end, color='#d62728', alpha=0.2,
                label='NBER Recession' if i == 0 else '')

ax1.set_ylabel('Recession Probability (%)')
ax1.set_ylim(0, 105)
ax1.legend(fontsize=9, loc='upper left')
ax1.set_title('NY Fed-Style Yield Curve Recession Model\nP(NBER Recession within 12 months)', fontweight='bold')

ax2.plot(df_plot.index, df_plot['yield_spread'], color='#2ca02c', lw=1.5, label='10Y-3M Yield Spread')
ax2.axhline(0, color='black', lw=1)
ax2.fill_between(df_plot.index, df_plot['yield_spread'], 0,
                  where=(df_plot['yield_spread'] < 0),
                  color='#d62728', alpha=0.3, label='Inverted (< 0)')
for start, end in recession_bands:
    ax2.axvspan(start, end, color='#d62728', alpha=0.15)

ax2.set_ylabel('Spread (pp)')
ax2.set_xlabel('Date')
ax2.legend(fontsize=9, loc='lower left')

plt.tight_layout()
plt.savefig('recession_probability_series.png', dpi=150, bbox_inches='tight')
plt.show()

Q1; 2022 to 2024 Inversion: At the deepest point of the inversion in late 2022/early 2023, the model predicted a recession probability of roughly 70 to 85%. period.

Q2; Interpreting a Miss: The model is not wrong. A predicted probability of 75% still means there was a 25% chance of no recession. Probabilistic forecasts communicate risk levels, not certainties. The 2022–2024 episode shows that even a strong, historically reliable signal can fail to materialize into a recession, other factors like strong labor markets and fiscal stimulus may have offset the contractionary signal. The correct takeaway is that high probability not equaled guaranteed outcome, and that no single variable model captures the full complexity of the economy.

Q3;2006 to 2007 Performance: The model performs well in this period. The yield curve inverted in 2006, and the model assigned steadily rising recession probabilities through 2007, correctly foreshadowing the 2008–2009 recession. This is widely regarded as one of the model's strongest historical forecasts and a key reason the NY Fed yield curve model maintains credibility despite the 2022–2024 miss.

SyntaxError: invalid character '–' (U+2013) (2369677846.py, line 179)